# GLOBAL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [ ]:
CLEAN_TMDB_FILE_PATH = "../datasets/clean/tmdb-movies/TMDB_movie_dataset_v11.csv"
CLEAN_MOVIELENS_RATINGS_PATH = "../datasets/clean/ml-32m/ratings.csv"

ML_API_TF_IDF_MATRIX_PATH = "../../ml-api/model/tf_idf_matrix.pkl"


tmdb_og = pd.read_csv(CLEAN_TMDB_FILE_PATH)
ratings_og = pd.read_csv(CLEAN_MOVIELENS_RATINGS_PATH)

In [ ]:
tmdb = tmdb_og
ratings = ratings_og

In [ ]:
def split_ratings_by_user(ratings_df, n, random_state=42):
    """
    Rozdělí dataset hodnocení na dvě části na základě unikátních uživatelů.
    
    :param ratings_df: Původní DataFrame s hodnoceními (musí obsahovat sloupec 'userId').
    :param n: Desetinné číslo v intervalu (0, 1) určující podíl uživatelů pro první tabulku.
    :param random_state: Seed pro reprodukovatelnost náhodného výběru.
    :return: Tuple dvou DataFrame (ratings_n, ratings_rest)
    """
    if not 0 < n < 1:
        raise ValueError("Parametr 'n' musí být v intervalu (0, 1).")

    # 1. Získáme pole všech unikátních uživatelů
    unique_users = ratings_df["userId"].unique()
    
    # 2. Vypočítáme, kolik jich má být v první (n%) skupině
    num_n_users = int(len(unique_users) * n)
    
    # 3. Náhodně vybereme přesně tento počet uživatelů (bez vracení)
    np.random.seed(random_state)
    users_n = np.random.choice(unique_users, size=num_n_users, replace=False)
    
    # Převedeme na set pro extrémně rychlé filtrování
    users_n_set = set(users_n)
    
    # 4. Rozštípnutí datasetu
    # Tabulka A: Pouze řádky uživatelů, kteří JSOU ve vybrané množině
    mask_n = ratings_df["userId"].isin(users_n_set)
    ratings_n = ratings_df[mask_n].copy()
    
    # Tabulka B: Pouze řádky uživatelů, kteří NEJSOU ve vybrané množině
    ratings_rest = ratings_df[~mask_n].copy()
    
    return ratings_n, ratings_rest

In [ ]:
def evaluate(
    build_user_profile, recommend_content, test_ratings, 
    threshold_rating=1.25, threshold_count=10, top_k=20, holdout_ratio=0.2
):
    """
    Rychlá offline evaluace doporučovacího systému.
    
    Implicitní hodnoty:
    - threshold_rating = 1.25: (Protože máš data vycentrovaná, 0.5 odpovídá hodnocení cca 3.75.0/5.0)
    - threshold_count = 10: Uživatel musí mít v profilu alespoň 10 filmů, aby měl model šanci ho pochopit.
    - top_k = 20: Standardní metrika Top-20 doporučení.
    - holdout_ratio = 0.2: Schováme 20 % z těch nadprůměrných filmů jako test.
    """
    
    hits = 0               # Uživatelé, u kterých jsme trefili alespoň 1 film
    total_precision = 0.0  # Suma precision pro výpočet průměru
    total_recall = 0.0     # Suma recall pro výpočet průměru
    total_sem_tfidf = 0.0  # Suma sémantické podobnosti přes TF-IDF texty
    total_sem_v = 0.       # Suma sémantické podobnosti přes SVD latentní prostor
    users_evaluated = 0

    # Shlukneme si data podle uživatelů
    grouped_users = test_ratings.groupby("userId")
    
    # Použijeme tqdm pro progress bar, ať vidíme, jak to běží (může to chvíli trvat)
    for user_id, user_data in tqdm(grouped_users, desc="Evaluace uživatelů"):
        
        # 1. Najdeme filmy, co se uživateli opravdu líbily
        liked_mask = user_data["rating"] >= threshold_rating
        liked_movies = user_data[liked_mask]
        
        # Pokud nemá žádný vysoko hodnocený film, nemáme co testovat
        if liked_movies.empty or len(liked_movies) < 2:
            continue
            
        # 2. Náhodně vybereme 'holdout_ratio' (např. 20%) z oblíbených filmů, které schováme
        num_to_hide = max(1, int(len(liked_movies) * holdout_ratio))
        
        hidden_movies = liked_movies.sample(n=num_to_hide, random_state=42)
        hidden_ids = set(hidden_movies["tmdbId"])
        
        # 3. Zbytek filmů (špatné + ty oblíbené, co jsme neschovali) použijeme pro profil
        train_movies = user_data[~user_data["tmdbId"].isin(hidden_ids)]
        
        if len(train_movies) < threshold_count:
            continue
            
        # 4. Vytvoříme slovník pro build_user_profile {tmdb_id: rating}
        # Tady použijeme čisté hodnocení, funkce si to už vyřeší podle své implementace
        train_dict = dict(zip(train_movies["tmdbId"], train_movies["rating"]))
        
        # 5. Spustíme tvé funkce (Model)
        user_profile = build_user_profile(train_dict)
        recs_df = recommend_content(user_profile, top_k=top_k)
        
        # Předpokládáme, že recs_df je DataFrame se sloupcem "id"
        rec_ids = set(recs_df["id"])
        
        # 6. Vyhodnocení (Zázrak se děje zde)
        # Průnik množin = kolik schovaných filmů se reálně objevilo v Top-K
        hits_in_top_k = len(hidden_ids.intersection(rec_ids))
        
        # Metriky
        precision = hits_in_top_k / top_k
        recall = hits_in_top_k / len(hidden_ids)
        
        if hits_in_top_k > 0:
            hits += 1
            
        total_precision += precision
        total_recall += recall

        # A) Podobnost přes TF-IDF (Textová shoda)
        h_idx_tfidf = [tmdb_id_to_index[h] for h in hidden_ids if h in tmdb_id_to_index]
        r_idx_tfidf = [tmdb_id_to_index[r] for r in rec_ids if r in tmdb_id_to_index]
        
        if h_idx_tfidf and r_idx_tfidf:
            # Vypočítáme matici podobností (hidden x recs) a pro každý skrytý film vezmeme MAX hodnotu
            sim_tfidf = cosine_similarity(tfidf_matrix[h_idx_tfidf], tfidf_matrix[r_idx_tfidf])
            total_sem_tfidf += sim_tfidf.max(axis=1).mean()

        # B) Podobnost přes matici V (Latentní/Behaviorální shoda)
        h_idx_v = [movie_map[h] for h in hidden_ids if h in movie_map]
        r_idx_v = [movie_map[r] for r in rec_ids if r in movie_map]
        
        if h_idx_v and r_idx_v:
            # Sloupce z V musíme transponovat na řádky pro cosine_similarity
            H_V = V[:, h_idx_v].T 
            R_V = V[:, r_idx_v].T
            sim_v = cosine_similarity(H_V, R_V)
            total_sem_v += sim_v.max(axis=1).mean()
        
        users_evaluated += 1

    # --- FINÁLNÍ VÝPIS ---
    if users_evaluated == 0:
        print("\nŽádný uživatel nesplnil tvé limity pro evaluaci!")
        return None
        
    avg_precision = total_precision / users_evaluated
    avg_recall = total_recall / users_evaluated
    hit_rate = hits / users_evaluated
    
    avg_sem_tfidf = total_sem_tfidf / users_evaluated
    avg_sem_v = total_sem_v / users_evaluated
    
    print("\n" + "="*45)
    print("=== OFFLINE EVALUATION RESULTS ===")
    print("="*45)
    print(f"Evaluated users: {users_evaluated}")
    print(f"Hit Rate@{top_k}:         {hit_rate:.4f} (Model found at least 1 movie for {hit_rate*100:.1f} % users)")
    print(f"Average Precision@{top_k}: {avg_precision:.4f}")
    print(f"Average Recall@{top_k}:    {avg_recall:.4f}")
    print("-" * 45)
    print(f"Semantic Recall (TF-IDF): {avg_sem_tfidf:.4f} (Avg top cosine similarity by tfidf)")
    print(f"Semantic Recall (SVD_V):  {avg_sem_v:.4f} (Avg top cosine similarity by latent features)")
    print("="*45)
    
    return {
        "hit_rate": hit_rate,
        "precision": avg_precision,
        "recall": avg_recall,
        "semantic_tfidf": avg_sem_tfidf,
        "semantic_v": avg_sem_v
    }

In [ ]:
movie_stats = ratings.groupby("tmdbId").agg(
    avg=("rating", "mean"),
    count=("rating", "count")
)
good_movies = set(movie_stats[movie_stats["count"] >= 0].index)
tmdb_ids = set(tmdb["id"])
valid_movie_ids = good_movies.intersection(tmdb_ids)
ratings = ratings[ratings["tmdbId"].isin(valid_movie_ids)].copy()
tmdb = tmdb[tmdb["id"].isin(valid_movie_ids)].copy()

tmdb = tmdb.reset_index(drop=True)
tmdb_id_to_index = pd.Series(tmdb.index, index=tmdb["id"]).to_dict()

ratings_glob_mean = 2.5
ratings["rating"] = ratings["rating"] - ratings_glob_mean

ratings_train, ratings_test = split_ratings_by_user(ratings, 0.93)

def find_in_dataset_by_substring(movie_names):
    found = []
    for movie_name in movie_names:
        results = tmdb[tmdb["title"].str.contains(movie_name, case=False, na=False)]
        haa = results.sort_values(by="popularity", ascending=False)[["id", "title"]]
        found.append(haa.values.tolist())
    return found

def find_in_dataset_by_id(movie_id):
    return tmdb[tmdb["id"] == movie_id]["title"].values[0]

def print_ratings_dict(ratings_dict: dict[int, int]):
    for id, rating in ratings_dict.items():
        print(f"id: {id}, name: {find_in_dataset_by_id(id)}, rating: {rating}")

In [ ]:
ratings["rating"].min(), ratings["rating"].max(), ratings["rating"].mean(), ratings["rating"].median(), ratings_glob_mean

In [ ]:
marvel_fan_rd = {
  569094: 4.5,
  634649: 4,
  271110: 3,
  1771: 4,
  10138: 4.5,
  1724: 3.5,
  299536: 5,
  9320: 4.5
}

print_ratings_dict(marvel_fan_rd)
print()
print()
print()

marvel_fan_dc_hater_rd = {
  569094: 4.5,
  634649: 4,
  271110: 3,
  1771: 4,
  10138: 4.5,
  1724: 3.5,
  299536: 5,
  9320: 4.5,
  141052: 1,
  209112: 0,
  414906: 2.5,
  272: 1.4,
  1924: 1,
  298618: 0.7,
  44912: 0,
  49521: 1.8,
  13183: 0.3,
  475557: 3
}

print_ratings_dict(marvel_fan_dc_hater_rd)

# CONTENT BASED

#### TRAINING

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(tmdb["content"])

#### METHODS

In [ ]:
def build_user_profile_content(ratings_dict):
    vectors = []
    weights = []

    for tmdb_id, rating in ratings_dict.items():
        if tmdb_id in tmdb_id_to_index:
            idx = tmdb_id_to_index[tmdb_id]
            vectors.append(tfidf_matrix[idx].toarray()[0])
            weights.append(rating - 2.5)

    vectors = np.array(vectors)
    weights = np.array(weights)

    user_profile = np.sum(vectors * weights[:, np.newaxis], axis=0)
    user_profile = user_profile.reshape(1, -1)

    return {
        "vector": user_profile,
        "method": "content"
    }

def recommend_content(user_vector, top_k=20):
    user_vector = user_vector["vector"]
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    weighted_sims = sims * tmdb["popularity_log"].values
    results = tmdb[["id", "title", "vote_average", "popularity"]].copy()
    results["match_score"] = weighted_sims

    # 1. Přidáme absolutní pořadí (1 = úplně nejlepší film pro tohoto uživatele)
    # ascending=False znamená, že nejvyšší skóre dostane rank 1
    results["rank"] = results["match_score"].rank(ascending=False).astype(int)
    
    # 2. Přidáme percentil (např. 99.5 znamená, že film je lepší než 99.5 % datasetu)
    # pct=True nám vrátí hodnoty od 0 do 1, takže to vynásobíme 100
    results["percentile"] = results["match_score"].rank(pct=True) * 100

    top_results = results.sort_values(by="match_score", ascending=False).head(top_k)

    return top_results

def find_recommended_content(user_vector, movie_id=1): # top_k tu asi nepotřebujeme, když hledáš konkrétní ID
    user_vector = user_vector["vector"]
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    weighted_sims = sims * tmdb["popularity_log"].values
    results = tmdb[["id", "title", "vote_average", "popularity"]].copy()
    results["match_score"] = weighted_sims

    # 1. Přidáme absolutní pořadí (1 = úplně nejlepší film pro tohoto uživatele)
    # ascending=False znamená, že nejvyšší skóre dostane rank 1
    results["rank"] = results["match_score"].rank(ascending=False).astype(int)
    
    # 2. Přidáme percentil (např. 99.5 znamená, že film je lepší než 99.5 % datasetu)
    # pct=True nám vrátí hodnoty od 0 do 1, takže to vynásobíme 100
    results["percentile"] = results["match_score"].rank(pct=True) * 100

    return results[results["id"] == movie_id]

def analyze_recommended_content(user_vector):
    user_vector = user_vector["vector"]
    sims = cosine_similarity(user_vector, tfidf_matrix).flatten()
    weighted_sims = sims * tmdb["popularity_log"].values
    
    results = tmdb[["id", "title", "vote_average", "popularity"]].copy()
    results["match_score"] = weighted_sims

    # --- 1. Základní statistiky a percentily ---
    print("=== STATISTIKY MATCH SCORE ===")
    stats = results["match_score"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])
    print(stats)
    print("\n")

    # --- 2. Analýza distribuce (Kladné vs Záporné) ---
    print("=== DISTRIBUCE SKÓRE ===")
    kladne = (results["match_score"] > 0).sum()
    zaporne = (results["match_score"] < 0).sum()
    nuly = (results["match_score"] == 0).sum()
    
    print(f"Filmy s kladným skóre (kandidáti na doporučení): {kladne}")
    print(f"Filmy se záporným skóre (aktivně penalizované):  {zaporne}")
    print(f"Filmy s nulovým skóre (žádná shoda v textu):    {nuly}")


#### EXAMPLE

In [ ]:
marvel_fan = build_user_profile_content(marvel_fan_rd)

#print(recommend_content(marvel_fan))
print(find_recommended_content(marvel_fan, movie_id = 791373))
print(find_recommended_content(marvel_fan, movie_id = 569094))
analyze_recommended_content(marvel_fan)

print()
print()
print()
print()

marvel_fan_dc_hater = build_user_profile_content(marvel_fan_dc_hater_rd)

#print(recommend_content(marvel_fan_dc_hater))
print(find_recommended_content(marvel_fan_dc_hater, movie_id = 791373))
print(find_recommended_content(marvel_fan_dc_hater, movie_id = 569094))
analyze_recommended_content(marvel_fan_dc_hater)

In [ ]:
recommend_content(build_user_profile_content(marvel_fan_rd))

In [ ]:
recommend_content(build_user_profile_content(marvel_fan_dc_hater_rd))

# COLLABORATIVE

#### TRAINING

In [ ]:
ratings_current = ratings_train.copy()


user_ids = np.sort(ratings_current["userId"].unique())
movie_ids = np.sort(ratings_current["tmdbId"].unique())

user_map = {u: i for i, u in enumerate(user_ids)}
movie_map = {m: i for i, m in enumerate(movie_ids)}

n_users = len(user_ids)
n_items = len(movie_ids)

rows = ratings_current["userId"].map(user_map)
cols = ratings_current["tmdbId"].map(movie_map)
data = ratings_current["rating"]

R = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))

svd = TruncatedSVD(n_components=35)
U = svd.fit_transform(R)
V = svd.components_

#### METHODS

In [ ]:
def build_user_profile_collab(ratings_dict):
    _indices = []
    _ratings = []

    for tmdb_id, rating in ratings_dict.items():
        if tmdb_id in movie_map:
            _indices.append(movie_map[tmdb_id])
            _ratings.append(rating - ratings_glob_mean)
        
    V_sub = V[:, _indices]
    r = np.array(_ratings)
    user_vector = r @ V_sub.T
    return {
        "vector": user_vector,
        "method": "collab"
    }

def recommend_collab(user_vector, top_k=20):
    user_vector = user_vector["vector"]
    # 1. Spočítáme skóre pro všechny filmy
    scores = user_vector @ V

    # 2. Vytvoříme si čistý DataFrame jen s IDčky a skóre
    score_df = pd.DataFrame({
        "id": movie_ids,
        "predicted_rating": scores
    })

    # 3. Bezpečně to spojíme s TMDB tabulkou
    results = tmdb[["id", "title", "vote_average", "popularity"]].merge(score_df, on="id")
    results["rank"] = results["predicted_rating"].rank(ascending=False).astype(int)
    results["percentile"] = results["predicted_rating"].rank(pct=True) * 100

    # 4. Seřadíme od nejvyššího skóre po nejnižší a vezmeme vrchních Top K
    top_results = results.sort_values(by="predicted_rating", ascending=False).head(top_k)

    return top_results

def find_recommended_collab(user_vector, movie_id):
    user_vector = user_vector["vector"]
    scores = user_vector @ V
    
    score_df = pd.DataFrame({
        "id": movie_ids,
        "predicted_rating": scores
    })
    
    results = tmdb[["id", "title", "vote_average", "popularity"]].merge(score_df, on="id")
    
    results["rank"] = results["predicted_rating"].rank(ascending=False).astype(int)
    results["percentile"] = results["predicted_rating"].rank(pct=True) * 100

    return results[results["id"] == movie_id]

def analyze_collab(user_vector):
    user_vector = user_vector["vector"]
    # 1. Výpočet skóre pomocí SVD matice
    scores = user_vector @ V
    
    # 2. Bezpečné napárování skóre k filmům (oprava bugu z minula)
    score_df = pd.DataFrame({
        "id": movie_ids,
        "predicted_rating": scores
    })
    
    # Spojení s TMDB tabulkou
    results = tmdb[["id", "title", "vote_average", "popularity"]].merge(score_df, on="id")

    # --- 1. Základní statistiky a percentily ---
    print("=== STATISTIKY PREDIKOVANÉHO HODNOCENÍ (SVD) ===")
    stats = results["predicted_rating"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])
    print(stats)
    print("\n")

    # --- 2. Analýza distribuce (Kladné vs Záporné) ---
    print("=== DISTRIBUCE SKÓRE ===")
    
    # Kladné skóre = Model očekává, že dáš filmu nadprůměrné hodnocení (např. > 3.5)
    positives = (results["predicted_rating"] > 0).sum()
    
    # Záporné skóre = Model očekává, že dáš filmu podprůměrné hodnocení (např. < 3.5)
    negatives = (results["predicted_rating"] < 0).sum()
    
    # Přesná nula se u maticových operací s float čísly objevuje naprosto výjimečně,
    # na rozdíl od TF-IDF, kde nula znamenala "žádné společné slovo".
    zeros = (results["predicted_rating"] == 0).sum()
    
    print(f"Filmy s kladným skóre (predikce NAD průměr): {positives}")
    print(f"Filmy se záporným skóre (predikce POD průměr): {negatives}")
    print(f"Filmy s nulovým skóre: {zeros}")
    
    return results

#### EXAMPLE

In [ ]:
marvel_fan = build_user_profile_collab(marvel_fan_rd)
marvel_fan_dc_hater = build_user_profile_collab(marvel_fan_dc_hater_rd)
#recommend_content(marvel_fan), recommend_content(marvel_fan_dc_hater)
print(find_recommended_collab(marvel_fan, movie_id = 791373))
print(find_recommended_collab(marvel_fan, movie_id = 569094))
analyze_collab(marvel_fan)

print()
print()
print()

print(find_recommended_collab(marvel_fan_dc_hater, movie_id = 791373))
print(find_recommended_collab(marvel_fan_dc_hater, movie_id = 569094))
analyze_collab(marvel_fan_dc_hater)


In [ ]:
recommend_collab(marvel_fan)

In [ ]:
recommend_collab(marvel_fan_dc_hater)

# COLLAB SIM

#### METHODS

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def build_user_profile_collab_sim(ratings_dict):
    """
    Vytvoří profil uživatele váženým součtem latentních vektorů oblíbených filmů.
    (Podobný princip jako u TF-IDF, ale s latentními vektory z matice V)
    """
    vectors = []
    weights = []

    for tmdb_id, rating in ratings_dict.items():
        if tmdb_id in movie_map:
            idx = movie_map[tmdb_id]
            # Vezmeme sloupec z matice V (latentní vektor filmu) a uděláme z něj 1D pole
            vectors.append(V[:, idx]) 
            # Opět posuneme hodnocení kolem středu (zde odečítáme např. globální průměr)
            weights.append(rating - ratings_glob_mean)

    if not vectors:
        # Fallback, pokud uživatel nehodnotil žádný existující film
        return np.zeros((1, V.shape[0]))

    vectors = np.array(vectors)
    weights = np.array(weights)

    # Vážený součet vektorů oblíbených filmů
    user_profile = np.sum(vectors * weights[:, np.newaxis], axis=0)
    # Reshape na 2D pole pro výpočet kosinové podobnosti
    user_profile = user_profile.reshape(1, -1)

    return {
        "vector": user_profile,
        "method": "collab sim"
    }

def recommend_collab_sim(user_vector, top_k=20):
    """
    Doporučí filmy na základě kosinové podobnosti uživatelského profilu 
    a latentních vektorů filmů (V.T).
    """
    user_vector = user_vector["vector"]
    # Spočítáme kosinovou podobnost mezi uživatelem a všemi filmy v latentním prostoru
    # V.T má tvar (počet filmů, 35). Výsledkem je flattenované 1D pole podobností.
    sims = cosine_similarity(user_vector, V.T).flatten()
    
    score_df = pd.DataFrame({
        "id": movie_ids,
        "predicted_rating": sims # Zde je skóre = kosinová podobnost
    })

    results = tmdb[["id", "title", "vote_average", "popularity"]].merge(score_df, on="id")
    results["rank"] = results["predicted_rating"].rank(ascending=False).astype(int)
    results["percentile"] = results["predicted_rating"].rank(pct=True) * 100

    top_results = results.sort_values(by="predicted_rating", ascending=False).head(top_k)

    return top_results

def find_recommended_collab_sim(user_vector, movie_id):
    """
    Najde skóre (kosinovou podobnost) pro konkrétní film.
    """
    user_vector = user_vector["vector"]
    sims = cosine_similarity(user_vector, V.T).flatten()
    
    score_df = pd.DataFrame({
        "id": movie_ids,
        "predicted_rating": sims
    })
    
    results = tmdb[["id", "title", "vote_average", "popularity"]].merge(score_df, on="id")
    results["rank"] = results["predicted_rating"].rank(ascending=False).astype(int)
    results["percentile"] = results["predicted_rating"].rank(pct=True) * 100

    return results[results["id"] == movie_id]

def analyze_collab_sim(user_vector):
    """
    Zanalyzuje rozložení skóre kosinové podobnosti v latentním prostoru.
    """
    user_vector = user_vector["vector"]
    sims = cosine_similarity(user_vector, V.T).flatten()
    
    score_df = pd.DataFrame({
        "id": movie_ids,
        "predicted_rating": sims
    })
    
    results = tmdb[["id", "title", "vote_average", "popularity"]].merge(score_df, on="id")

    print("=== STATISTIKY KOSINOVÉ PODOBNOSTI (Lantentní prostor) ===")
    stats = results["predicted_rating"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])
    print(stats)
    print("\n")

    print("=== DISTRIBUCE SKÓRE ===")
    positives = (results["predicted_rating"] > 0).sum()
    negatives = (results["predicted_rating"] < 0).sum()
    zeros = (results["predicted_rating"] == 0).sum()
    
    print(f"Filmy s kladnou podobností (stejný směr v latentním prostoru): {positives}")
    print(f"Filmy se zápornou podobností (opačný směr): {negatives}")
    print(f"Filmy s nulovou podobností (ortogonální): {zeros}")
    
    return results

#### EXAMPLES

In [ ]:
marvel_fan = build_user_profile_collab_sim(marvel_fan_rd)
marvel_fan_dc_hater = build_user_profile_collab_sim(marvel_fan_dc_hater_rd)
#recommend_content(marvel_fan), recommend_content(marvel_fan_dc_hater)
print(find_recommended_collab_sim(marvel_fan, movie_id = 791373))
print(find_recommended_collab_sim(marvel_fan, movie_id = 569094))
analyze_collab_sim(marvel_fan)

print()
print()
print()

print(find_recommended_collab_sim(marvel_fan_dc_hater, movie_id = 791373))
print(find_recommended_collab_sim(marvel_fan_dc_hater, movie_id = 569094))
analyze_collab_sim(marvel_fan_dc_hater)

In [ ]:
recommend_collab_sim(marvel_fan)

In [ ]:
recommend_collab_sim(marvel_fan_dc_hater)

# HYBRID

#### METHODS

In [ ]:
def build_user_profile_hybrid(ratings_dict, niche_threshold=1000):
    """
    Vytvoří profil uživatele a určí, zda je uživatel fanoušek blockbusterů (CF),
    nebo spíše niche/artových filmů (CBF).
    
    :param niche_threshold: Hranice průměrného počtu hodnocení (vote_count). 
                            Pod touto hranicí se zapne Content-Based.
    """
    # 1. Zjistíme, jak moc jsou filmy v profilu známé
    vote_counts = []
    
    # Použijeme náš mapovací slovník tmdb_id_to_index z dřívějška
    for tmdb_id in ratings_dict.keys():
        if tmdb_id in tmdb_id_to_index:
            idx = tmdb_id_to_index[tmdb_id]
            # Vezmeme počet hodnocení přímo z TMDB tabulky (sloupec vote_count)
            votes = tmdb.loc[idx, "vote_count"]
            vote_counts.append(votes)
            
    # Pokud nemáme žádné validní filmy, spadneme bezpečně do CF
    if not vote_counts:
        return {
            "vector": build_user_profile_collab(ratings_dict),
            "method": "collab"
        }
        
    # Použijeme medián, aby jeden obrovský blockbuster (např. Avengers) 
    # nepřevážil 10 neznámých filmů.
    median_votes = np.median(vote_counts)
    
    # 2. DETERMINISTICKÉ ROZHODNUTÍ
    if median_votes < niche_threshold:
        # Uživatel má rád neznámé filmy -> Spouštíme textový model
        return {
            "vector": build_user_profile_content(ratings_dict),
            "method": "content"
        }
    else:
        # Uživatel kouká na mainstream -> Spouštíme SVD
        return {
            "vector": build_user_profile_collab(ratings_dict),
            "method": "collab"
        }

def recommend_hybrid(user_profile_obj, top_k=20):
    """
    Přijme objekt profilu a zavolá adekvátní doporučovací funkci.
    """
    method = user_profile_obj["method"]
    user_vector = user_profile_obj["vector"]
    
    if method == "content":
        #print(f"(Spouštím Content-Based Filtering | Niche divák)")
        return recommend_content(user_vector, top_k=top_k)
        
    elif method == "collab":
        #print(f"(Spouštím Collaborative Filtering | Mainstream divák)")
        return recommend_collab(user_vector, top_k=top_k)
        
    else:
        raise ValueError(f"Neznámá metoda: {method}")

# FIRST EVAL

In [ ]:
evaluate(
    build_user_profile_content, recommend_content, ratings_test
)

In [ ]:
evaluate(
    build_user_profile_collab, recommend_collab, ratings_test
)

In [ ]:
evaluate(
    build_user_profile_collab_sim, recommend_collab_sim, ratings_test
)

In [ ]:
evaluate(
    build_user_profile_hybrid, recommend_hybrid, ratings_test
)